# AMPS I · Week 15 — Lab
## Steady Navier–Stokes: Newton, Reynolds-number continuation, the lid-driven cavity, PCD and H(div)-DG

| | |
|---|---|
| **Lecture sections** | Part A (Newton vs Picard, continuation, PCD, H(div)-DG) · Part B (code) · Part C (measurements) · Part D (LLM) |
| **Time** | ≈ 80 min |
| **You will** | solve the cavity with Newton and compare with Ghia et al. (1982) · watch Newton fail at Re = 1000 without continuation · compare Picard and Newton · measure GMRES iterations with a mass-matrix and a PCD Schur complement · see an exactly divergence-free velocity |

Unit square, lid $\mathbf u=(1,0)$ on $y=1$, $\mathbf u=0$ elsewhere, $\mathrm{Re}=1/\nu$; Taylor–Hood unless stated. Ghia's benchmark values are in the cells (`GHIA_*`).

**Reminder.** `%%fenicsx` cells run in a separate MPI process; ordinary cells have NumPy and matplotlib only. Data passes through `/tmp/fnx/`.

## ⚙️ Setup — run once per runtime
Installs DOLFINx 0.11 and registers the `%%fenicsx` cell magic. Re-run after every runtime restart.

In [ ]:
# ══════════════════════════════════════════════════════════════════
#  FEniCSx on Google Colab — run once per runtime (≈ 3–5 min)
#  https://github.com/SeoulTechPSE/fenicsx-colab
# ══════════════════════════════════════════════════════════════════
from google.colab import drive
import os, multiprocessing, subprocess
from pathlib import Path

if not os.path.ismount('/content/drive'):
    drive.mount('/content/drive')

REPO_URL = 'https://github.com/seoultechpse/fenicsx-colab.git'
REPO_DIR = Path('/content/fenicsx-colab')
if not REPO_DIR.exists():
    subprocess.run(['git', 'clone', REPO_URL, str(REPO_DIR)], check=True)

# OpenMPI 5.x slot patch: Colab has 2 CPUs; allow up to 4 ranks (oversubscribed)
N_PROC = 4
with open('/root/hostfile', 'w') as f:
    f.write(f'localhost slots={N_PROC}\n')
os.environ['OMPI_MCA_rmaps_default_mapping_policy'] = 'slot:OVERSUBSCRIBE'
os.environ['PRTE_MCA_rmaps_default_mapping_policy'] = 'slot:OVERSUBSCRIBE'
os.environ['OMPI_MCA_osc'] = '^ucx'   # skip the UCX one-sided module (harmless start-up message)

DOLFINX_VERSION = '0.11'
USE_COMPLEX = False
USE_CLEAN   = False
opts = [f'--version {DOLFINX_VERSION}']
if USE_COMPLEX: opts.append('--complex')
if USE_CLEAN:   opts.append('--clean')
get_ipython().run_line_magic('run', f"{REPO_DIR / 'setup_fenicsx.py'} {' '.join(opts)}")

# Folder shared by %%fenicsx cells (computation) and ordinary cells (plotting)
os.makedirs('/tmp/fnx', exist_ok=True)

In [ ]:
%%fenicsx --info -np 2

## 1 · Newton for the cavity at Re = 100  *(lecture §A2, §B1–B2)*
$N=32$, `crossed` mesh. Read `cavity` first: the residual `F` with `Re` as a `fem.Constant`, the pressure pinned at the origin, MUMPS inside Newton, a monitor that records the residuals.

In [ ]:
%%fenicsx -np 2
from mpi4py import MPI
from petsc4py import PETSc
import time
import numpy as np
import ufl
import basix.ufl
import dolfinx.fem.petsc
from dolfinx import fem, mesh, geometry
from dolfinx.fem.petsc import LinearProblem, NonlinearProblem

comm = MPI.COMM_WORLD

def say(*a):
    if comm.rank == 0:
        print(*a, flush=True)
# Ghia, Ghia & Shin (1982), Tables I and II (transcription: gist ivan-pi 3e9326d1..., caa6c673...)
GHIA_Y = np.array([1.0000, 0.9766, 0.9688, 0.9609, 0.9531, 0.8516, 0.7344, 0.6172, 0.5000, 0.4531, 0.2813, 0.1719, 0.1016, 0.0703, 0.0625, 0.0547, 0.0000])
GHIA_U = {100: [1.0, 0.84123, 0.78871, 0.73722, 0.68717, 0.23151, 0.00332, -0.13641, -0.20581, -0.21090, -0.15662, -0.10150, -0.06434, -0.04775, -0.04192, -0.03717, 0.0],
          400: [1.0, 0.75837, 0.68439, 0.61756, 0.55892, 0.29093, 0.16256, 0.02135, -0.11477, -0.17119, -0.32726, -0.24299, -0.14612, -0.10338, -0.09266, -0.08186, 0.0],
          1000: [1.0, 0.65928, 0.57492, 0.51117, 0.46604, 0.33304, 0.18719, 0.05702, -0.06080, -0.10648, -0.27805, -0.38289, -0.29730, -0.22220, -0.20196, -0.18109, 0.0]}
GHIA_X = np.array([1.0000, 0.9688, 0.9609, 0.9531, 0.9453, 0.9063, 0.8594, 0.8047, 0.5000, 0.2344, 0.2266, 0.1563, 0.0938, 0.0781, 0.0703, 0.0625, 0.0000])
GHIA_V = {100: [0.0, -0.05906, -0.07391, -0.08864, -0.10313, -0.16914, -0.22445, -0.24533, 0.05454, 0.17527, 0.17507, 0.16077, 0.12317, 0.10890, 0.10091, 0.09233, 0.0],
          400: [0.0, -0.12146, -0.15663, -0.19254, -0.22847, -0.23827, -0.44993, -0.38598, 0.05186, 0.30174, 0.30203, 0.28124, 0.22965, 0.20920, 0.19713, 0.18360, 0.0],
          1000: [0.0, -0.21388, -0.27669, -0.33714, -0.39188, -0.51500, -0.42665, -0.31966, 0.02526, 0.32235, 0.33075, 0.37095, 0.32627, 0.30353, 0.29012, 0.27485, 0.0]}

TH = basix.ufl.mixed_element([basix.ufl.element("Lagrange", "triangle", 2, shape=(2,)), basix.ufl.element("Lagrange", "triangle", 1)])

def cavity(N, diag="crossed", picard=False, max_it=30):
    """Steady lid-driven cavity, Taylor-Hood, residual F(w) = 0 with Re a fem.Constant. Newton (or Picard: picard=True) + MUMPS.
    Returns msh, w, Re, problem, residual history (filled by a SNES monitor)."""
    msh = mesh.create_unit_square(comm, N, N, diagonal=getattr(mesh.DiagonalType, diag))
    W = fem.functionspace(msh, TH)
    W0, _ = W.sub(0).collapse(); W1, _ = W.sub(1).collapse()
    lid = fem.Function(W0)
    lid.interpolate(lambda x: np.vstack([np.isclose(x[1], 1.0).astype(float), np.zeros(x.shape[1])]))
    msh.topology.create_connectivity(1, 2)
    bf = mesh.exterior_facet_indices(msh.topology)
    bcs = [fem.dirichletbc(lid, fem.locate_dofs_topological((W.sub(0), W0), 1, bf), W.sub(0)),
           fem.dirichletbc(fem.Function(W1), fem.locate_dofs_geometrical((W.sub(1), W1), lambda x: np.isclose(x[0], 0) & np.isclose(x[1], 0)), W.sub(1))]
    w = fem.Function(W); Re = fem.Constant(msh, 100.0)
    u, p = ufl.split(w); v, q = ufl.TestFunctions(W)
    F = (1 / Re * ufl.inner(ufl.grad(u), ufl.grad(v)) + ufl.inner(ufl.dot(ufl.grad(u), u), v) - p * ufl.div(v) - q * ufl.div(u)) * ufl.dx
    J = None                                    # None: NonlinearProblem differentiates F (Newton)
    if picard:                                  # Oseen: the convecting velocity u is frozen at the current iterate
        du, dp = ufl.TrialFunctions(W)
        J = (1 / Re * ufl.inner(ufl.grad(du), ufl.grad(v)) + ufl.inner(ufl.dot(ufl.grad(du), u), v) - dp * ufl.div(v) - q * ufl.div(du)) * ufl.dx
    pr = NonlinearProblem(F, w, bcs=bcs, J=J, petsc_options_prefix=f"cav{N}{diag}{int(picard)}_",
                          petsc_options={"snes_type": "newtonls", "snes_linesearch_type": "none", "snes_rtol": 1e-10, "snes_atol": 1e-12,
                                         "snes_max_it": max_it, "ksp_type": "preonly", "pc_type": "lu", "pc_factor_mat_solver_type": "mumps"})
    hist = []
    pr.solver.setMonitor(lambda snes, it, rnorm: hist.append(rnorm))
    return msh, w, Re, pr, hist

def profiles(msh, w, ys=None, xs=None):
    """u(0.5, y) and v(x, 0.5) (default: at Ghia's points). Point evaluation in parallel: the owning rank evaluates, Allreduce(MAX)."""
    u = w.sub(0).collapse()
    tree = geometry.bb_tree(msh, 2)
    def ev(P, comp):
        P = np.c_[P, np.zeros(len(P))]
        cells = geometry.compute_colliding_cells(msh, geometry.compute_collisions_points(tree, P), P)
        out = np.full(len(P), -np.inf)
        for i in range(len(P)):
            c = cells.links(i)
            if len(c):
                out[i] = u.eval(P[i], c[:1])[comp]
        res = np.empty_like(out); comm.Allreduce(out, res, op=MPI.MAX)
        return res
    ys = GHIA_Y if ys is None else ys
    xs = GHIA_X if xs is None else xs
    return ev(np.c_[np.full(len(ys), 0.5), ys], 0), ev(np.c_[xs, np.full(len(xs), 0.5)], 1)

SUSPECT = 5      # GHIA_X[5] = 0.9063: Ghia's Re = 400 value there is flagged as a probable misprint

msh, w, Re, pr, hist = cavity(32)
pr.solve()
say(f"Re = 100: SNES reason {pr.solver.getConvergedReason()}, {pr.solver.getIterationNumber()} Newton iterations, residuals {[f'{h:.1e}' for h in hist]}")
uy, vx = profiles(msh, w)
say(f"max |u - Ghia| = {np.abs(uy - GHIA_U[100]).max():.4f},  max |v - Ghia| = {np.abs(vx - GHIA_V[100]).max():.4f}")
yy = np.linspace(0, 1, 101); U, Vv = profiles(msh, w, yy, yy)
if comm.rank == 0:
    np.savez("/tmp/fnx/w15_re100.npz", yy=yy, U=U, V=Vv)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

d = np.load("/tmp/fnx/w15_re100.npz")
GY = [1.0, 0.9766, 0.9688, 0.9609, 0.9531, 0.8516, 0.7344, 0.6172, 0.5, 0.4531, 0.2813, 0.1719, 0.1016, 0.0703, 0.0625, 0.0547, 0.0]
GU = [1.0, 0.84123, 0.78871, 0.73722, 0.68717, 0.23151, 0.00332, -0.13641, -0.20581, -0.2109, -0.15662, -0.1015, -0.06434, -0.04775, -0.04192, -0.03717, 0.0]
fig, ax = plt.subplots(figsize=(4.5, 4.5))
ax.plot(d["U"], d["yy"], label="Taylor-Hood, N = 32"); ax.plot(GU, GY, "o", mfc="none", label="Ghia et al. (1982)")
ax.set_xlabel("u(0.5, y)"); ax.set_ylabel("y"); ax.legend(); fig.tight_layout(); plt.show()

> **✏️ Your turn (10 min).** Run the same with **Picard** (`cavity(32, picard=True, max_it=60)`) for Re = 100 from zero, then Re = 400 from that solution. Print the residual
> histories of Picard and Newton side by side. Which convergence order do you see, and why is one Picard iteration cheaper than one Newton iteration only in theory here?

In [ ]:
%%fenicsx -np 2
# TODO: Picard vs Newton

## 2 · Reynolds-number continuation  *(lecture §A3, §C1)*
The same mesh, Re = 100 → 400 → 1000, each solve starting from the previous solution — and each starting from zero.

In [ ]:
%%fenicsx -np 2
from mpi4py import MPI
from petsc4py import PETSc
import time
import numpy as np
import ufl
import basix.ufl
import dolfinx.fem.petsc
from dolfinx import fem, mesh, geometry
from dolfinx.fem.petsc import LinearProblem, NonlinearProblem

comm = MPI.COMM_WORLD

def say(*a):
    if comm.rank == 0:
        print(*a, flush=True)
# Ghia, Ghia & Shin (1982), Tables I and II (transcription: gist ivan-pi 3e9326d1..., caa6c673...)
GHIA_Y = np.array([1.0000, 0.9766, 0.9688, 0.9609, 0.9531, 0.8516, 0.7344, 0.6172, 0.5000, 0.4531, 0.2813, 0.1719, 0.1016, 0.0703, 0.0625, 0.0547, 0.0000])
GHIA_U = {100: [1.0, 0.84123, 0.78871, 0.73722, 0.68717, 0.23151, 0.00332, -0.13641, -0.20581, -0.21090, -0.15662, -0.10150, -0.06434, -0.04775, -0.04192, -0.03717, 0.0],
          400: [1.0, 0.75837, 0.68439, 0.61756, 0.55892, 0.29093, 0.16256, 0.02135, -0.11477, -0.17119, -0.32726, -0.24299, -0.14612, -0.10338, -0.09266, -0.08186, 0.0],
          1000: [1.0, 0.65928, 0.57492, 0.51117, 0.46604, 0.33304, 0.18719, 0.05702, -0.06080, -0.10648, -0.27805, -0.38289, -0.29730, -0.22220, -0.20196, -0.18109, 0.0]}
GHIA_X = np.array([1.0000, 0.9688, 0.9609, 0.9531, 0.9453, 0.9063, 0.8594, 0.8047, 0.5000, 0.2344, 0.2266, 0.1563, 0.0938, 0.0781, 0.0703, 0.0625, 0.0000])
GHIA_V = {100: [0.0, -0.05906, -0.07391, -0.08864, -0.10313, -0.16914, -0.22445, -0.24533, 0.05454, 0.17527, 0.17507, 0.16077, 0.12317, 0.10890, 0.10091, 0.09233, 0.0],
          400: [0.0, -0.12146, -0.15663, -0.19254, -0.22847, -0.23827, -0.44993, -0.38598, 0.05186, 0.30174, 0.30203, 0.28124, 0.22965, 0.20920, 0.19713, 0.18360, 0.0],
          1000: [0.0, -0.21388, -0.27669, -0.33714, -0.39188, -0.51500, -0.42665, -0.31966, 0.02526, 0.32235, 0.33075, 0.37095, 0.32627, 0.30353, 0.29012, 0.27485, 0.0]}

TH = basix.ufl.mixed_element([basix.ufl.element("Lagrange", "triangle", 2, shape=(2,)), basix.ufl.element("Lagrange", "triangle", 1)])

def cavity(N, diag="crossed", picard=False, max_it=30):
    """Steady lid-driven cavity, Taylor-Hood, residual F(w) = 0 with Re a fem.Constant. Newton (or Picard: picard=True) + MUMPS.
    Returns msh, w, Re, problem, residual history (filled by a SNES monitor)."""
    msh = mesh.create_unit_square(comm, N, N, diagonal=getattr(mesh.DiagonalType, diag))
    W = fem.functionspace(msh, TH)
    W0, _ = W.sub(0).collapse(); W1, _ = W.sub(1).collapse()
    lid = fem.Function(W0)
    lid.interpolate(lambda x: np.vstack([np.isclose(x[1], 1.0).astype(float), np.zeros(x.shape[1])]))
    msh.topology.create_connectivity(1, 2)
    bf = mesh.exterior_facet_indices(msh.topology)
    bcs = [fem.dirichletbc(lid, fem.locate_dofs_topological((W.sub(0), W0), 1, bf), W.sub(0)),
           fem.dirichletbc(fem.Function(W1), fem.locate_dofs_geometrical((W.sub(1), W1), lambda x: np.isclose(x[0], 0) & np.isclose(x[1], 0)), W.sub(1))]
    w = fem.Function(W); Re = fem.Constant(msh, 100.0)
    u, p = ufl.split(w); v, q = ufl.TestFunctions(W)
    F = (1 / Re * ufl.inner(ufl.grad(u), ufl.grad(v)) + ufl.inner(ufl.dot(ufl.grad(u), u), v) - p * ufl.div(v) - q * ufl.div(u)) * ufl.dx
    J = None                                    # None: NonlinearProblem differentiates F (Newton)
    if picard:                                  # Oseen: the convecting velocity u is frozen at the current iterate
        du, dp = ufl.TrialFunctions(W)
        J = (1 / Re * ufl.inner(ufl.grad(du), ufl.grad(v)) + ufl.inner(ufl.dot(ufl.grad(du), u), v) - dp * ufl.div(v) - q * ufl.div(du)) * ufl.dx
    pr = NonlinearProblem(F, w, bcs=bcs, J=J, petsc_options_prefix=f"cav{N}{diag}{int(picard)}_",
                          petsc_options={"snes_type": "newtonls", "snes_linesearch_type": "none", "snes_rtol": 1e-10, "snes_atol": 1e-12,
                                         "snes_max_it": max_it, "ksp_type": "preonly", "pc_type": "lu", "pc_factor_mat_solver_type": "mumps"})
    hist = []
    pr.solver.setMonitor(lambda snes, it, rnorm: hist.append(rnorm))
    return msh, w, Re, pr, hist

def profiles(msh, w, ys=None, xs=None):
    """u(0.5, y) and v(x, 0.5) (default: at Ghia's points). Point evaluation in parallel: the owning rank evaluates, Allreduce(MAX)."""
    u = w.sub(0).collapse()
    tree = geometry.bb_tree(msh, 2)
    def ev(P, comp):
        P = np.c_[P, np.zeros(len(P))]
        cells = geometry.compute_colliding_cells(msh, geometry.compute_collisions_points(tree, P), P)
        out = np.full(len(P), -np.inf)
        for i in range(len(P)):
            c = cells.links(i)
            if len(c):
                out[i] = u.eval(P[i], c[:1])[comp]
        res = np.empty_like(out); comm.Allreduce(out, res, op=MPI.MAX)
        return res
    ys = GHIA_Y if ys is None else ys
    xs = GHIA_X if xs is None else xs
    return ev(np.c_[np.full(len(ys), 0.5), ys], 0), ev(np.c_[xs, np.full(len(xs), 0.5)], 1)

SUSPECT = 5      # GHIA_X[5] = 0.9063: Ghia's Re = 400 value there is flagged as a probable misprint

msh, w, Re, pr, hist = cavity(32)
for mode in ("continuation", "from zero"):
    w.x.array[:] = 0.0
    for re in (100, 400, 1000):
        if mode == "from zero":
            w.x.array[:] = 0.0
        Re.value = re; pr.solve()
        r, its = pr.solver.getConvergedReason(), pr.solver.getIterationNumber()
        uy, vx = profiles(msh, w)
        say(f"{mode:12s} Re = {re:4d}: reason {r:3d}, {its:2d} Newton its | max|u - Ghia| {np.abs(uy - GHIA_U[re]).max():.3g}")

> **✏️ Your turn (5 min).** Another way to reach Re = 1000 from zero: a few **Picard** steps (they converge from far away, slowly) followed by Newton (fast, but only
> close to the solution). Do 3 Picard steps, copy the result into a Newton problem and continue. How many Newton iterations are needed?

In [ ]:
%%fenicsx -np 2
# TODO: Picard start, then Newton, Re = 1000 from zero

## 3 · Krylov solvers inside Newton: the mass matrix vs PCD  *(lecture §A4, §B3, §C3)*
With LU we never saw the linear algebra. Now GMRES with the Week 12 block preconditioner — velocity block by LU, Schur complement approximated either by
$M_p/\nu$ (Week 12) or by **PCD** from `fenicsx-pctools`. First install the package (Colab, about 20 s).

In [ ]:
# fenicsx-pctools (PCD preconditioner) into the %%fenicsx environment on Colab. That environment has no pip: bootstrap it first.
# --no-deps keeps DOLFINx 0.11 untouched (the package declares dolfinx <= 0.11.0.dev0 but works with 0.11.0). Not on Colab: skipped.
import os, subprocess
PY = "/content/micromamba/envs/fenicsx/bin/python"
if os.path.exists(PY):
    run = lambda cmd: subprocess.run(cmd, capture_output=True, text=True)
    if run([PY, "-c", "import pip"]).returncode != 0:
        print("ensurepip:", "ok" if run([PY, "-m", "ensurepip", "--upgrade"]).returncode == 0 else "FAILED")
    r = run([PY, "-m", "pip", "install", "--no-deps", "-q", "git+https://gitlab.com/rafinex-external-rifle/fenicsx-pctools.git"])
    print("fenicsx-pctools:", "ok" if r.returncode == 0 else "FAILED\n" + r.stderr[-1500:])
else:
    print("not on Colab: install fenicsx-pctools yourself (pip install --no-deps git+https://gitlab.com/rafinex-external-rifle/fenicsx-pctools.git)")

In [ ]:
%%fenicsx -np 2
from mpi4py import MPI
from petsc4py import PETSc
import time
import numpy as np
import ufl
import basix.ufl
import dolfinx.fem.petsc
from dolfinx import fem, mesh, geometry
from dolfinx.fem.petsc import LinearProblem, NonlinearProblem

comm = MPI.COMM_WORLD

def say(*a):
    if comm.rank == 0:
        print(*a, flush=True)
# Ghia, Ghia & Shin (1982), Tables I and II (transcription: gist ivan-pi 3e9326d1..., caa6c673...)
GHIA_Y = np.array([1.0000, 0.9766, 0.9688, 0.9609, 0.9531, 0.8516, 0.7344, 0.6172, 0.5000, 0.4531, 0.2813, 0.1719, 0.1016, 0.0703, 0.0625, 0.0547, 0.0000])
GHIA_U = {100: [1.0, 0.84123, 0.78871, 0.73722, 0.68717, 0.23151, 0.00332, -0.13641, -0.20581, -0.21090, -0.15662, -0.10150, -0.06434, -0.04775, -0.04192, -0.03717, 0.0],
          400: [1.0, 0.75837, 0.68439, 0.61756, 0.55892, 0.29093, 0.16256, 0.02135, -0.11477, -0.17119, -0.32726, -0.24299, -0.14612, -0.10338, -0.09266, -0.08186, 0.0],
          1000: [1.0, 0.65928, 0.57492, 0.51117, 0.46604, 0.33304, 0.18719, 0.05702, -0.06080, -0.10648, -0.27805, -0.38289, -0.29730, -0.22220, -0.20196, -0.18109, 0.0]}
GHIA_X = np.array([1.0000, 0.9688, 0.9609, 0.9531, 0.9453, 0.9063, 0.8594, 0.8047, 0.5000, 0.2344, 0.2266, 0.1563, 0.0938, 0.0781, 0.0703, 0.0625, 0.0000])
GHIA_V = {100: [0.0, -0.05906, -0.07391, -0.08864, -0.10313, -0.16914, -0.22445, -0.24533, 0.05454, 0.17527, 0.17507, 0.16077, 0.12317, 0.10890, 0.10091, 0.09233, 0.0],
          400: [0.0, -0.12146, -0.15663, -0.19254, -0.22847, -0.23827, -0.44993, -0.38598, 0.05186, 0.30174, 0.30203, 0.28124, 0.22965, 0.20920, 0.19713, 0.18360, 0.0],
          1000: [0.0, -0.21388, -0.27669, -0.33714, -0.39188, -0.51500, -0.42665, -0.31966, 0.02526, 0.32235, 0.33075, 0.37095, 0.32627, 0.30353, 0.29012, 0.27485, 0.0]}

from fenicsx_pctools.mat import create_splittable_matrix_block
from fenicsx_pctools.pc import PCDPC_vX

class MassPC(PCDPC_vX):
    """Schur complement ~ -(M_p/nu)^-1: the Stokes choice of Week 12, blind to convection."""
    _work_vecs = ()                                   # nothing to free in destroy()

    def apply(self, pc, x, y):
        self.ksp_Mp.solve(x, y); y.scale(-1.0)

def cavity_krylov(N, Res, schur, diag="crossed"):
    """Newton for the cavity with GMRES + block upper-triangular Schur preconditioner: velocity block by LU, Schur complement by
    PCD (schur="PCD") or by the pressure mass matrix (schur="mass"). Re continuation over Res. Returns [(Re, reason, Newton its, GMRES its)]."""
    msh = mesh.create_unit_square(comm, N, N, diagonal=getattr(mesh.DiagonalType, diag))
    V = fem.functionspace(msh, ("Lagrange", 2, (2,))); Q = fem.functionspace(msh, ("Lagrange", 1))
    Wm = ufl.MixedFunctionSpace(V, Q)
    u, p = fem.Function(V), fem.Function(Q); v, q = ufl.TestFunctions(Wm)
    nu = fem.Constant(msh, 1.0 / Res[0])
    F = (ufl.inner(ufl.dot(ufl.grad(u), u), v) + nu * ufl.inner(ufl.grad(u), ufl.grad(v)) - p * ufl.div(v) - q * ufl.div(u)) * ufl.dx
    J = ufl.derivative(F, (u, p), ufl.TrialFunctions(Wm))
    F_dfx, J_dfx = fem.form(ufl.extract_blocks(F)), fem.form(ufl.extract_blocks(J))
    lid = fem.Function(V); lid.interpolate(lambda x: np.vstack([np.isclose(x[1], 1.0).astype(float), np.zeros(x.shape[1])]))
    msh.topology.create_connectivity(1, 2)
    bcs = [fem.dirichletbc(lid, fem.locate_dofs_topological(V, 1, mesh.exterior_facet_indices(msh.topology)))]
    J_mat = dolfinx.fem.petsc.assemble_matrix(J_dfx, bcs); J_mat.assemble()
    Js = create_splittable_matrix_block(J_mat, ufl.extract_blocks(J), nu=nu, v=u)   # no bcs_pcd: A_p gets the constant null space
    F_vec = dolfinx.fem.petsc.create_vector([V, Q], kind="mpi"); x0 = dolfinx.fem.petsc.create_vector([V, Q], kind="mpi")
    nv = x0.duplicate(); nv.set(0.0); nv.array[V.dofmap.index_map.size_local * 2:] = 1.0; nv.normalize()
    nsp = PETSc.NullSpace().create(vectors=[nv], comm=comm); Js.setNullSpace(nsp); J_mat.setNullSpace(nsp)   # enclosed flow: p up to a constant
    def Ffun(snes, x, b):
        dolfinx.fem.petsc.assemble_residual(snes, x, b, [u, p], F_dfx, J_dfx, bcs, _blocks=F_vec.getAttr("_blocks"))
    def Jfun(snes, x, Jm, P):
        Jm.zeroEntries(); dolfinx.fem.petsc.assemble_matrix(J_mat, J_dfx, bcs, diag=1.0); Jm.assemble()
    o = PETSc.Options(); pre = f"kry{schur}{N}{diag}_"
    opts = {"snes_type": "newtonls", "snes_linesearch_type": "basic", "snes_rtol": 1e-8, "snes_atol": 1e-10, "snes_max_it": 30,
            "ksp_type": "gmres", "ksp_gmres_restart": 300, "ksp_pc_side": "right", "ksp_rtol": 1e-6, "ksp_max_it": 1000,
            "pc_type": "python", "pc_python_type": "fenicsx_pctools.pc.WrappedPC",
            "wrapped_pc_type": "fieldsplit", "wrapped_pc_fieldsplit_type": "schur", "wrapped_pc_fieldsplit_schur_fact_type": "upper",
            "wrapped_pc_fieldsplit_schur_precondition": "user", "wrapped_pc_fieldsplit_0_fields": 0, "wrapped_pc_fieldsplit_1_fields": 1,
            "wrapped_fieldsplit_0_ksp_type": "preonly", "wrapped_fieldsplit_0_pc_type": "lu", "wrapped_fieldsplit_0_pc_factor_mat_solver_type": "mumps",
            "wrapped_fieldsplit_1_ksp_type": "preonly", "wrapped_fieldsplit_1_pc_type": "python",
            "wrapped_fieldsplit_1_pc_python_type": "fenicsx_pctools.pc.PCDPC_vX" if schur == "PCD" else "__main__.MassPC",
            "wrapped_fieldsplit_1_pcd_Mp_ksp_type": "preonly", "wrapped_fieldsplit_1_pcd_Mp_pc_type": "lu",
            "wrapped_fieldsplit_1_pcd_Ap_ksp_type": "preonly", "wrapped_fieldsplit_1_pcd_Ap_pc_type": "hypre"}
    for k, val in opts.items():
        o[pre + k] = val
    snes = PETSc.SNES().create(comm); snes.setFunction(Ffun, F_vec); snes.setJacobian(Jfun, J=Js, P=None)
    snes.setOptionsPrefix(pre); snes.setFromOptions()
    out = []
    for Re_ in Res:
        nu.value = 1.0 / Re_
        snes.solve(None, x0)
        out.append((Re_, snes.getConvergedReason(), snes.getIterationNumber(), snes.getLinearSolveIterations()))
        if out[-1][1] < 0:
            break
    snes.destroy(); Js.destroy(); J_mat.destroy()                 # destroy the Python preconditioners before the interpreter exits
    for k in opts:
        o.delValue(pre + k)
    return out

for schur, Res in (("mass", (100, 400)), ("PCD", (100, 400, 1000))):
    for (re, r, nit, lin) in cavity_krylov(32, Res, schur):
        say(f"S ~ {schur:4s} Re = {re:4d}: SNES reason {r}, {nit} Newton, {lin:4d} GMRES ({lin / max(nit, 1):5.1f} per Newton step)")

import gc; gc.collect(); PETSc.garbage_cleanup()

> **✏️ Your turn (5 min).** Repeat PCD on $N=64$ (Re = 100, 400). Do the GMRES counts per Newton step depend on the mesh? On Re? What does that say about how well
> $M_p/\nu$ and the PCD operator approximate the Schur complement $BF^{-1}B^\top$ (lecture §A4)?

In [ ]:
%%fenicsx -np 2
# TODO: PCD on N = 64

## 4 · An exactly divergence-free velocity: H(div)-conforming DG  *(lecture §A5, §B4, §C4)*
Kovasznay flow (exact steady NS solution, Re = 25). The DOLFINx 0.11 demo `demo_navier-stokes.py`: velocity in Raviart–Thomas RT2, pressure in DG1, symmetric interior
penalty for the viscous term, upwinding for convection, steady state by 25 implicit time steps. Compare with Taylor–Hood Newton on the same meshes.

In [ ]:
%%fenicsx -np 2
from mpi4py import MPI
from petsc4py import PETSc
import time
import numpy as np
import ufl
import basix.ufl
import dolfinx.fem.petsc
from dolfinx import fem, mesh, geometry
from dolfinx.fem.petsc import LinearProblem, NonlinearProblem

comm = MPI.COMM_WORLD

def say(*a):
    if comm.rank == 0:
        print(*a, flush=True)
# Ghia, Ghia & Shin (1982), Tables I and II (transcription: gist ivan-pi 3e9326d1..., caa6c673...)
GHIA_Y = np.array([1.0000, 0.9766, 0.9688, 0.9609, 0.9531, 0.8516, 0.7344, 0.6172, 0.5000, 0.4531, 0.2813, 0.1719, 0.1016, 0.0703, 0.0625, 0.0547, 0.0000])
GHIA_U = {100: [1.0, 0.84123, 0.78871, 0.73722, 0.68717, 0.23151, 0.00332, -0.13641, -0.20581, -0.21090, -0.15662, -0.10150, -0.06434, -0.04775, -0.04192, -0.03717, 0.0],
          400: [1.0, 0.75837, 0.68439, 0.61756, 0.55892, 0.29093, 0.16256, 0.02135, -0.11477, -0.17119, -0.32726, -0.24299, -0.14612, -0.10338, -0.09266, -0.08186, 0.0],
          1000: [1.0, 0.65928, 0.57492, 0.51117, 0.46604, 0.33304, 0.18719, 0.05702, -0.06080, -0.10648, -0.27805, -0.38289, -0.29730, -0.22220, -0.20196, -0.18109, 0.0]}
GHIA_X = np.array([1.0000, 0.9688, 0.9609, 0.9531, 0.9453, 0.9063, 0.8594, 0.8047, 0.5000, 0.2344, 0.2266, 0.1563, 0.0938, 0.0781, 0.0703, 0.0625, 0.0000])
GHIA_V = {100: [0.0, -0.05906, -0.07391, -0.08864, -0.10313, -0.16914, -0.22445, -0.24533, 0.05454, 0.17527, 0.17507, 0.16077, 0.12317, 0.10890, 0.10091, 0.09233, 0.0],
          400: [0.0, -0.12146, -0.15663, -0.19254, -0.22847, -0.23827, -0.44993, -0.38598, 0.05186, 0.30174, 0.30203, 0.28124, 0.22965, 0.20920, 0.19713, 0.18360, 0.0],
          1000: [0.0, -0.21388, -0.27669, -0.33714, -0.39188, -0.51500, -0.42665, -0.31966, 0.02526, 0.32235, 0.33075, 0.37095, 0.32627, 0.30353, 0.29012, 0.27485, 0.0]}

ReK = 25.0
lamK = ReK / 2 - np.sqrt(ReK**2 / 4 + 4 * np.pi**2)

def kov_exact(x, m=ufl):
    u = (1 - m.exp(lamK * x[0]) * m.cos(2 * np.pi * x[1]), lamK / (2 * np.pi) * m.exp(lamK * x[0]) * m.sin(2 * np.pi * x[1]))
    return (ufl.as_vector(u) if m is ufl else np.vstack(u)), 0.5 * (1 - m.exp(2 * lamK * x[0]))

def hdg_kovasznay(n, k=1, num_steps=25, t_end=10.0):
    msh = mesh.create_unit_square(comm, n, n)
    V = fem.functionspace(msh, ("Raviart-Thomas", k + 1)); Q = fem.functionspace(msh, ("Discontinuous Lagrange", k))
    VQ = ufl.MixedFunctionSpace(V, Q)
    u, p = ufl.TrialFunctions(VQ); v, q = ufl.TestFunctions(VQ)
    dt = fem.Constant(msh, t_end / num_steps); alpha = fem.Constant(msh, 6.0 * k**2)
    h = ufl.CellDiameter(msh); nn = ufl.FacetNormal(msh)
    jump = lambda phi: ufl.outer(phi("+"), nn("+")) + ufl.outer(phi("-"), nn("-"))
    a = (1.0 / ReK) * (ufl.inner(ufl.grad(u), ufl.grad(v)) * ufl.dx - ufl.inner(ufl.avg(ufl.grad(u)), jump(v)) * ufl.dS
                       - ufl.inner(jump(u), ufl.avg(ufl.grad(v))) * ufl.dS + (alpha / ufl.avg(h)) * ufl.inner(jump(u), jump(v)) * ufl.dS
                       - ufl.inner(ufl.grad(u), ufl.outer(v, nn)) * ufl.ds - ufl.inner(ufl.outer(u, nn), ufl.grad(v)) * ufl.ds
                       + (alpha / h) * ufl.inner(ufl.outer(u, nn), ufl.outer(v, nn)) * ufl.ds)
    a -= ufl.inner(p, ufl.div(v)) * ufl.dx + ufl.inner(ufl.div(u), q) * ufl.dx
    u_D = fem.Function(V); u_D.interpolate(lambda x: kov_exact(x, np)[0])
    L = (1 / ReK) * (-ufl.inner(ufl.outer(u_D, nn), ufl.grad(v)) * ufl.ds + (alpha / h) * ufl.inner(ufl.outer(u_D, nn), ufl.outer(v, nn)) * ufl.ds)
    L += ufl.inner(fem.Constant(msh, 0.0), q) * ufl.dx
    msh.topology.create_connectivity(1, 2)
    bcs = [fem.dirichletbc(u_D, fem.locate_dofs_topological(V, 1, mesh.exterior_facet_indices(msh.topology)))]
    opts = {"ksp_type": "preonly", "pc_type": "lu", "pc_factor_mat_solver_type": "mumps", "mat_mumps_icntl_14": 80,
            "mat_mumps_icntl_24": 1, "mat_mumps_icntl_25": 0, "ksp_error_if_not_converged": 1}
    u_h, p_h = fem.Function(V), fem.Function(Q)
    LinearProblem(ufl.extract_blocks(a), ufl.extract_blocks(L), u=[u_h, p_h], bcs=bcs, kind="mpi", petsc_options_prefix=f"hdgS{n}_", petsc_options=opts).solve()
    u_n = fem.Function(V); u_n.x.array[:] = u_h.x.array
    lmbda = ufl.conditional(ufl.gt(ufl.dot(u_n, nn), 0), 1, 0)          # upwind indicator
    u_uw = lmbda("+") * u("+") + lmbda("-") * u("-")
    a += (ufl.inner(u / dt, v) * ufl.dx - ufl.inner(u, ufl.div(ufl.outer(v, u_n))) * ufl.dx + ufl.inner((ufl.dot(u_n, nn))("+") * u_uw, v("+")) * ufl.dS
          + ufl.inner((ufl.dot(u_n, nn))("-") * u_uw, v("-")) * ufl.dS + ufl.inner(ufl.dot(u_n, nn) * lmbda * u, v) * ufl.ds)
    L += ufl.inner(u_n / dt, v) * ufl.dx - ufl.inner(ufl.dot(u_n, nn) * (1 - lmbda) * u_D, v) * ufl.ds
    ns = LinearProblem(ufl.extract_blocks(a), ufl.extract_blocks(L), u=[u_h, p_h], bcs=bcs, kind="mpi", petsc_options_prefix=f"hdgN{n}_", petsc_options=opts)
    for _ in range(num_steps):
        ns.solve(); u_n.x.array[:] = u_h.x.array
    x = ufl.SpatialCoordinate(msh); ue, _ = kov_exact(x)
    S = lambda e: np.sqrt(comm.allreduce(fem.assemble_scalar(fem.form(e * ufl.dx(metadata={"quadrature_degree": 8}))), op=MPI.SUM))
    return S(ufl.inner(u_h - ue, u_h - ue)), S(ufl.div(u_h)**2)

for n in (8, 16, 32):
    eu, ed = hdg_kovasznay(n)
    say(f"H(div)-DG, n = {n:2d}: |u - u_h| = {eu:.3e},  |div u_h| = {ed:.1e}")

> **✏️ Your turn (5 min).** Solve Kovasznay with Taylor–Hood and Newton (adapt `cavity`: exact velocity on the whole boundary, Re = 25) for $n=8,16,32$ and print
> $\|\mathbf u-\mathbf u_h\|$ and $\|\nabla\cdot\mathbf u_h\|$. Which method is more accurate per mesh, which one conserves mass exactly, and why (lecture §A5)?

In [ ]:
%%fenicsx -np 2
# TODO: Taylor-Hood on Kovasznay

## 5 · LLM-assisted coding: live demo  *(lecture §D1)*
Prompt (paste the API reference sheet first):

```text
Context: [API reference sheet]
Task: DOLFINx 0.11, one %%fenicsx -np 2 cell. Steady lid-driven cavity on the unit square (lid velocity (1, 0) on y = 1), Taylor-Hood
elements, Newton's method with NonlinearProblem and MUMPS. Use Reynolds-number continuation 100 -> 400 -> 1000 and print, for each
Re, the number of Newton iterations and the u-velocity at (0.5, 0.2813) and (0.5, 0.1719) (Ghia et al. 1982: -0.15662, -0.10150 at
Re = 100; -0.32726, -0.24299 at Re = 400; -0.27805, -0.38289 at Re = 1000).
```
Paste the result below **unchanged**, run it, then go through the checklist.

In [ ]:
%%fenicsx -np 2
# TODO: paste the %%fenicsx part your LLM produced (unchanged) and run it

**Verification checklist** — checks with a known answer

- [ ] Convection in the residual is $(\mathbf u\cdot\nabla)\mathbf u$ (Week 14: `dot(grad(u), u)`)
- [ ] The pressure constant is fixed (a pinned value or a null space)
- [ ] Re = 100 agrees with Ghia within about $10^{-2}$ on $N=32$
- [ ] Newton converges quadratically (look at the residuals)
- [ ] Same numbers with `-np 1` and `-np 2`